In [1]:
import pandas as pd
import pickle
from mlxtend.frequent_patterns import association_rules

## Pattern mining on purchases and different sections

This section of the analysis aims to show the most frequent patterns in purchase products of different sections in the same session. The purpose for this is to show a link to a particular section when the user adds to the basket a product of another section, in order to push the user to buy more products.

In [2]:
with open("frequent_purchase_section_2.pkl", "rb") as f:  # "rb" = read binary
    s_purchases = pickle.load(f)

s_purchases = pd.DataFrame(s_purchases)

print("Number of frequent sections")
print(len(s_purchases))

Number of frequent sections
141


In [3]:
s_purchases_rules = association_rules(s_purchases, metric="confidence", min_threshold=0.3)
s_purchases_rules = s_purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]

print("Association rules with minimum confidence level set to 0.3")
s_purchases_rules

Association rules with minimum confidence level set to 0.3


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ computers, accessories})",frozenset({ appliances}),0.005587,0.436083,0.002456,0.439573,1.008003
1,"frozenset({ appliances, accessories})",frozenset({ computers}),0.006463,0.350288,0.002456,0.379981,1.084767
2,"frozenset({ accessories, electronics})",frozenset({ computers}),0.003549,0.350288,0.001193,0.336283,0.960020
3,"frozenset({ accessories, electronics})",frozenset({ appliances}),0.003549,0.436083,0.001278,0.360177,0.825937
4,frozenset({ electronics}),frozenset({ appliances}),0.123306,0.436083,0.040101,0.325218,0.745770
...,...,...,...,...,...,...,...
78,"frozenset({ construction, sport})",frozenset({ computers}),0.007154,0.350288,0.002453,0.342845,0.978752
79,"frozenset({ computers, construction, sport})",frozenset({ appliances}),0.002453,0.436083,0.001482,0.604353,1.385868
80,"frozenset({ construction, appliances, sport})",frozenset({ computers}),0.003332,0.350288,0.001482,0.444863,1.269994
81,"frozenset({ construction, country_yard})",frozenset({ appliances}),0.009069,0.436083,0.005555,0.612535,1.404629


Using confidence as threshold value to determine the best association rules is not so reliable in this situation. This is because there are some sections (particularly "appliances" and "computers") with a really high support. The computation of confidence takes into account only how frequent the antecedent is, causing an inflation of the measure. This can also be seen when looking at the lift, which for some rules is even below 1. This is why, in this situation, is much more better to compute association rules using a minimum lift threshold.

In [4]:
s_purchases_rules = association_rules(s_purchases, metric="lift", min_threshold=2)
s_purchases_rules = s_purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]

print("Association rules with minimum lift level set to 2")
s_purchases_rules

Association rules with minimum lift level set to 2


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ computers, sport})","frozenset({ appliances, electronics})",0.039282,0.040101,0.003505,0.089216,2.224763
1,"frozenset({ appliances, sport})","frozenset({ computers, electronics})",0.043367,0.034643,0.003505,0.080811,2.332649
2,"frozenset({ electronics, sport})","frozenset({ computers, appliances})",0.013089,0.104244,0.003505,0.267754,2.568525
3,"frozenset({ computers, appliances})","frozenset({ electronics, sport})",0.104244,0.013089,0.003505,0.033619,2.568525
4,"frozenset({ computers, electronics})","frozenset({ appliances, sport})",0.034643,0.043367,0.003505,0.101160,2.332649
5,"frozenset({ appliances, electronics})","frozenset({ computers, sport})",0.040101,0.039282,0.003505,0.087392,2.224763
6,"frozenset({ furniture, appliances})","frozenset({ computers, sport})",0.017586,0.039282,0.001394,0.079286,2.018392
7,"frozenset({ furniture, sport})","frozenset({ computers, appliances})",0.005618,0.104244,0.001394,0.248183,2.380784
8,"frozenset({ computers, appliances})","frozenset({ furniture, sport})",0.104244,0.005618,0.001394,0.013375,2.380784
9,"frozenset({ computers, sport})","frozenset({ furniture, appliances})",0.039282,0.017586,0.001394,0.035494,2.018392


Using the lift to compute association rules causes another problem. The lift is a measure that can be highly inflated when using really rare itemsets. This happens because the denominator is the product of the support measures of the two itemsets. In this situation probably the best strategy is to set minimum threshold values for both the measures.

In [5]:
s_purchases_rules = association_rules(s_purchases, metric="lift", min_threshold=2)
s_purchases_rules = s_purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]
s_purchases_rules = s_purchases_rules[s_purchases_rules["confidence"] > 0.10]
s_purchases_rules = s_purchases_rules.sort_values("lift", ascending=False)
s_purchases_rules = s_purchases_rules.reset_index(drop=True)
s_purchases_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ computers, medicine})","frozenset({ appliances, electronics})",0.016800,0.040101,0.001878,0.111776,2.787338
1,"frozenset({ computers, construction})","frozenset({ appliances, electronics})",0.015626,0.040101,0.001743,0.111535,2.781345
2,"frozenset({ computers, country_yard})","frozenset({ appliances, sport})",0.017520,0.043367,0.001963,0.112027,2.583230
3,"frozenset({ electronics, sport})","frozenset({ computers, appliances})",0.013089,0.104244,0.003505,0.267754,2.568525
4,"frozenset({ furniture, electronics})","frozenset({ computers, appliances})",0.005637,0.104244,0.001445,0.256267,2.458333
5,"frozenset({ furniture, sport})","frozenset({ computers, appliances})",0.005618,0.104244,0.001394,0.248183,2.380784
6,"frozenset({ medicine, electronics})","frozenset({ computers, appliances})",0.007675,0.104244,0.001878,0.244681,2.347185
7,"frozenset({ computers, electronics})","frozenset({ appliances, sport})",0.034643,0.043367,0.003505,0.101160,2.332649
8,"frozenset({ construction, electronics})","frozenset({ computers, appliances})",0.007176,0.104244,0.001743,0.242888,2.329990
9,"frozenset({ computers, medicine})","frozenset({ appliances, sport})",0.016800,0.043367,0.001686,0.100374,2.314514


### Conclusions

The rules above should suggest possible purchase patterns followed by the users. When an user buys one or more products of the antecedent section / sections, there is a reasonable possibility that he will buy one or more product also of the consequent section / sections.
The advised strategy is set as follows:
- Let's take as an example the first rule, so {computers, medicine} -> {electronics, appliances};
- Let's assume an user has already added to its basket one product from "computers" section and one from "medicine" section;
- After the user adds to the basket this last product, the page should show a fast link to the "electronics" section and one to the "appliances" section.

## Pattern mining on purchases and products

The purpose of this section is to find strong associations between purchases of products, in order to provide an efficient advise policy and encourage the user to buy more products.

In [6]:
with open("frequent_purchases.pkl", "rb") as f:  # "rb" = read binary
    purchases = pickle.load(f)

purchases = pd.DataFrame(purchases)

print("Number of frequent itemsets")
print(len(purchases))

Number of frequent itemsets
524


In [7]:
purchases_rules = association_rules(purchases, metric="confidence", min_threshold=0.4)
purchases_rules = purchases_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]
print("Association rules with minimum confidence level set to 0.4")
purchases_rules

Association rules with minimum confidence level set to 0.4


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,frozenset({214820233}),frozenset({214820261}),0.003407,0.005524,0.002283,0.670046,121.303101
1,frozenset({214820261}),frozenset({214820233}),0.005524,0.003407,0.002283,0.413303,121.303101
2,frozenset({214716710}),frozenset({214716707}),0.001928,0.003583,0.001212,0.628664,175.455247
3,frozenset({214834877}),frozenset({214834880}),0.010419,0.010328,0.004491,0.430983,41.728123
4,frozenset({214834880}),frozenset({214834877}),0.010328,0.010419,0.004491,0.434783,41.728123
5,frozenset({214826610}),frozenset({214834880}),0.009468,0.010328,0.004924,0.520066,50.353300
6,frozenset({214834880}),frozenset({214826610}),0.010328,0.009468,0.004924,0.476741,50.353300
7,frozenset({214826610}),frozenset({214834877}),0.009468,0.010419,0.004098,0.432836,41.541281
8,"frozenset({214834877, 214826610})",frozenset({214834880}),0.004098,0.010328,0.002371,0.578544,56.015167
9,"frozenset({214834877, 214834880})",frozenset({214826610}),0.004491,0.009468,0.002371,0.527972,55.764353


In [8]:
purchases_rules_no = association_rules(purchases, metric="lift", min_threshold=0.0000000001)
print("Number of association rules with no threshold value")
print(len(purchases_rules_no))
purchases_rules_l = association_rules(purchases, metric="lift", min_threshold=2)
print("Number of patterns with threshold value of the lift set equal to 1.5")
print(len(purchases_rules_l))

Number of association rules with no threshold value
216
Number of patterns with threshold value of the lift set equal to 1.5
216


# Conclusions

In this situation the confidence is a more accurate measure since the lift is highly inflated by not-so-frequent itemsets. This is easily showed by the fact that the number of association rules does not change with or without a normal threshold value for the lift (2 should already be a "good" association).

The advised strategy is to show a link / advertisement for the consequent product / products when the antecedent product / products are added to the basket.

In [9]:
purchases_rules = purchases_rules[["antecedents", "consequents", "antecedent support", "consequent support", "support", "confidence", "lift"]]
mono_associations = purchases_rules[(purchases_rules["antecedents"].apply(len) == 1) & (purchases_rules["consequents"].apply(len) == 1)]
multi_associations = purchases_rules[(purchases_rules["antecedents"].apply(len) > 1) | (purchases_rules["consequents"].apply(len) > 1)]

print(mono_associations.head())
print(multi_associations)

              antecedents             consequents  antecedent support  \
0  frozenset({214820233})  frozenset({214820261})            0.003407   
1  frozenset({214820261})  frozenset({214820233})            0.005524   
2  frozenset({214716710})  frozenset({214716707})            0.001928   
3  frozenset({214834877})  frozenset({214834880})            0.010419   
4  frozenset({214834880})  frozenset({214834877})            0.010328   

   consequent support   support  confidence        lift  
0            0.005524  0.002283    0.670046  121.303101  
1            0.003407  0.002283    0.413303  121.303101  
2            0.003583  0.001212    0.628664  175.455247  
3            0.010328  0.004491    0.430983   41.728123  
4            0.010419  0.004491    0.434783   41.728123  
                          antecedents             consequents  \
8   frozenset({214834877, 214826610})  frozenset({214834880})   
9   frozenset({214834877, 214834880})  frozenset({214826610})   
10  frozenset({214

In [10]:
purchases_rules["pair"] = purchases_rules.apply(
    lambda r: tuple(sorted([
        next(iter(r["antecedents"])),
        next(iter(r["consequents"]))
    ])),
    axis=1
)
pair_counts = purchases_rules["pair"].value_counts()
asymmetric_pairs = pair_counts[pair_counts == 1]
print("Mono-directional relationships")
asymmetric_pairs

Mono-directional relationships


pair
(214716707, 214716710)    1
(214833755, 214834877)    1
(214826610, 214833755)    1
(214826610, 214829387)    1
(214829387, 214834880)    1
(214829387, 214834877)    1
(214829861, 214829865)    1
(214829846, 214829865)    1
(214829878, 214829887)    1
(214829885, 214831950)    1
(214835017, 214835019)    1
(214829765, 214835109)    1
(214835109, 214835167)    1
(214835109, 214836932)    1
(214826990, 214838227)    1
(214829765, 214836924)    1
(214835167, 214836924)    1
(214839971, 214839973)    1
(214837286, 214838092)    1
(214844355, 214844370)    1
(214848337, 214848380)    1
Name: count, dtype: int64

For an association rule {A} -> {B} that respects the confidence threshold value let's define it as:
- Bi-directional rule: if the relation {B} -> {A} also respects the confidence threshold value;
- Mono-directional rule: if the relation {B} -> {A} does not respect the confidence threshold value.

The last output shows all the mono-directional rules regarding purchases with only one product as antecedent (the first one) and only one product as consequent (the second one). The business strategy of above suggests to show the link / advertisement only when the first product is added to the basket, but another strategy could be implemented for the opposite relation.

Assuming the rule to be {A} -> {B}, let's assume that in general it is really likely to buy the two products together, but the customer is more likely to remember to buy product B after buying product A, but not to buy product A after buying product B. After implementing a correct A/B test to check for this hypothesis, an efficient strategy would be to show advertiments for both the directions, in order to increase purchases of the products together, independently of which one is bought first.

## Pattern mining on clicks in different sections

This section of the analysis aims to show the most frequent patterns in clicking products of different sections in the same session. The purpose for this is to show a link to a particular section when the user clicks on a product of another section, in order to push the user to buy more products.

In [2]:
with open("frequent_clicks_sections_2.pkl", "rb") as f:  # "rb" = read binary
    s_clicks = pickle.load(f)

s_clicks = pd.DataFrame(s_clicks)

print("Number of frequent sections")
print(len(s_clicks))

Number of frequent sections
933


In this situation the minimum support was set equal to 0.005 because the clicks are much more frequent

In [3]:
s_clicks_rules = association_rules(s_clicks, metric="confidence", min_threshold=0.5)
s_clicks_rules = s_clicks_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]

print("Association rules with minimum confidence level set to 0.9")
s_clicks_rules

Association rules with minimum confidence level set to 0.9


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ electronics, computers})",frozenset({ appliances}),0.081344,0.535649,0.046854,0.576002,1.075334
1,frozenset({ furniture}),frozenset({ appliances}),0.128933,0.535649,0.068234,0.529223,0.988002
2,"frozenset({ electronics, furniture})",frozenset({ appliances}),0.030507,0.535649,0.019720,0.646410,1.206778
3,"frozenset({ electronics, furniture})",frozenset({ computers}),0.030507,0.433499,0.017631,0.577930,1.333176
4,"frozenset({ electronics, furniture, applianc...",frozenset({ computers}),0.019720,0.433499,0.013514,0.685312,1.580886
...,...,...,...,...,...,...,...
1627,"frozenset({ sport, appliances})",frozenset({ computers}),0.083310,0.433499,0.042063,0.504902,1.164714
1628,"frozenset({ electronics, sport})",frozenset({ appliances}),0.034614,0.535649,0.021049,0.608109,1.135274
1629,"frozenset({ electronics, sport})",frozenset({ computers}),0.034614,0.433499,0.020167,0.582628,1.344014
1630,"frozenset({ electronics, computers, sport})",frozenset({ appliances}),0.020167,0.535649,0.014624,0.725167,1.353810


The clicks are more frequent than the purchases, consequently even with a confidence level set to 0.9 there are 166 association rules. As before, in such a situation, the confidence could be inflated by some really frequent sections (i.e. appliances just as before, with a support of 0.53). So using the lift could be an efficient strategy.

In [4]:
s_clicks_rules = association_rules(s_clicks, metric="lift", min_threshold=2)
s_clicks_rules = s_clicks_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]

print("Association rules with minimum lift level set to 4")
s_clicks_rules

Association rules with minimum lift level set to 4


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ electronics, computers, applianc...",frozenset({ furniture}),0.046854,0.128933,0.013514,0.288430,2.237048
1,"frozenset({ furniture, appliances, computers})",frozenset({ electronics}),0.033792,0.188770,0.013514,0.399926,2.118592
2,"frozenset({ electronics, furniture})","frozenset({ computers, appliances})",0.030507,0.205260,0.013514,0.442992,2.158206
3,"frozenset({ electronics, appliances})","frozenset({ furniture, computers})",0.093973,0.057349,0.013514,0.143809,2.507596
4,"frozenset({ electronics, computers})","frozenset({ furniture, appliances})",0.081344,0.068234,0.013514,0.166136,2.434787
...,...,...,...,...,...,...,...
12271,frozenset({ sport}),"frozenset({ medicine, auto, appliances})",0.180245,0.002299,0.001012,0.005617,2.443375
12272,"frozenset({ electronics, computers})","frozenset({ sport, appliances})",0.081344,0.083310,0.014624,0.179784,2.158023
12273,"frozenset({ electronics, sport})","frozenset({ computers, appliances})",0.034614,0.205260,0.014624,0.422503,2.058384
12274,"frozenset({ computers, appliances})","frozenset({ electronics, sport})",0.205260,0.034614,0.014624,0.071248,2.058384


Just as before for the purchases, the lift alone shows some problems too, being inflated by really rare sections. So the best strategy is using threshold values for both the measures.

In [6]:
s_clicks_rules = association_rules(s_clicks, metric="lift", min_threshold=2)
s_clicks_rules = s_clicks_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]
s_clicks_rules = s_clicks_rules[s_clicks_rules["confidence"] > 0.60]
s_clicks_rules = s_clicks_rules.sort_values("lift", ascending=False)
s_clicks_rules = s_clicks_rules.reset_index(drop=True)
s_clicks_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"frozenset({ apparel, furniture, accessories,...","frozenset({ electronics, computers})",0.001543,0.081344,0.001021,0.661739,8.135062
1,"frozenset({ sport, accessories, appliances, ...","frozenset({ electronics, computers})",0.001785,0.081344,0.001135,0.635928,7.817754
2,"frozenset({ auto, furniture, appliances, co...","frozenset({ electronics, computers})",0.001741,0.081344,0.001101,0.632689,7.777929
3,"frozenset({ furniture, accessories, applianc...","frozenset({ electronics, computers})",0.002260,0.081344,0.001422,0.628970,7.732214
4,"frozenset({ furniture, accessories, applianc...","frozenset({ electronics, computers})",0.002030,0.081344,0.001277,0.628882,7.731136
...,...,...,...,...,...,...,...
409,"frozenset({ electronics, appliances, kids, ...",frozenset({ computers}),0.001153,0.433499,0.001005,0.871292,2.009906
410,"frozenset({ electronics, apparel, sport, co...",frozenset({ computers}),0.001351,0.433499,0.001175,0.869972,2.006862
411,"frozenset({ electronics, apparel, furniture,...",frozenset({ computers}),0.001178,0.433499,0.001024,0.869440,2.005635
412,"frozenset({ electronics, apparel, country_ya...",frozenset({ computers}),0.001174,0.433499,0.001020,0.869004,2.004629


### Conclusions

The results show itemsets with a lot of sections that do not seem correlated. Even with high threshold values for both the measures, the number of association rules computed is really big. Consequently we can assess that probably these results are not really reliable, but just the direct consequence of a wide navigation in the website of the company, maybe without a clear willing to buy certain products from different sections.

## Pattern mining on clicks and products

The purpose of this section is to find strong associations between clicks on products, in order to provide an efficient advise policy and encourage the user to buy more products.

In [7]:
with open("frequent_clicks.pkl", "rb") as f:  # "rb" = read binary
    clicks = pickle.load(f)

clicks = pd.DataFrame(clicks)

clicks_rules = association_rules(clicks, metric="confidence", min_threshold=0.2)
print(len(clicks_rules))

99


In [8]:
clicks_rules = association_rules(clicks, metric="confidence", min_threshold=0.4)
clicks_rules = clicks_rules[["antecedents", "consequents", "antecedent support",
                                       "consequent support", "support", "confidence", "lift"]]
print("Association rules with minimum confidence level set to 0.4")
clicks_rules

Association rules with minimum confidence level set to 0.4


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,frozenset({214838227}),frozenset({214835109}),0.003541,0.005359,0.001707,0.481891,89.930089
1,frozenset({214819762}),frozenset({214819760}),0.003582,0.004821,0.001717,0.479364,99.442240
2,frozenset({214826912}),frozenset({214828987}),0.002927,0.003162,0.001475,0.503956,159.377916
3,frozenset({214828987}),frozenset({214826912}),0.003162,0.002927,0.001475,0.466479,159.377916
4,frozenset({214834865}),frozenset({214827007}),0.002141,0.002877,0.001037,0.484193,168.298295
5,frozenset({214834865}),frozenset({214827005}),0.002141,0.003044,0.001014,0.473378,155.488926
6,frozenset({214827000}),frozenset({214827005}),0.001942,0.003044,0.001083,0.557798,183.218256
7,frozenset({214584992}),frozenset({214840595}),0.002341,0.001749,0.001010,0.431507,246.666364
8,frozenset({214840595}),frozenset({214584992}),0.001749,0.002341,0.001010,0.577393,246.666364
9,frozenset({214831948}),frozenset({214829878}),0.002635,0.013804,0.001185,0.449628,32.571783


In [10]:
clicks_rules_no = association_rules(clicks, metric="lift", min_threshold=0.0000000001)
print("Number of association rules with no threshold value")
print(len(clicks_rules_no))
clicks_rules_l = association_rules(clicks, metric="lift", min_threshold=2)
print("Number of patterns with threshold value of the lift set equal to 1.5")
print(len(clicks_rules_l))

Number of association rules with no threshold value
134
Number of patterns with threshold value of the lift set equal to 1.5
134


This data behaves in the same way of data about purchases. The lift is not really reliable, since it is inflated by the really rare products. Confidence gives more accurate and reliable association rules.

The advised strategy is to show a link / advertisement for the consequent product / products after clicking on the antecedent product / products.

In [11]:
clicks_rules = clicks_rules[["antecedents", "consequents", "antecedent support", "consequent support", "support", "confidence", "lift"]]
mono_associations = clicks_rules[(clicks_rules["antecedents"].apply(len) == 1) & (clicks_rules["consequents"].apply(len) == 1)]
multi_associations = clicks_rules[(clicks_rules["antecedents"].apply(len) > 1) | (clicks_rules["consequents"].apply(len) > 1)]

print(mono_associations.head())
print(multi_associations)

              antecedents             consequents  antecedent support  \
0  frozenset({214838227})  frozenset({214835109})            0.003541   
1  frozenset({214819762})  frozenset({214819760})            0.003582   
2  frozenset({214826912})  frozenset({214828987})            0.002927   
3  frozenset({214828987})  frozenset({214826912})            0.003162   
4  frozenset({214834865})  frozenset({214827007})            0.002141   

   consequent support   support  confidence        lift  
0            0.005359  0.001707    0.481891   89.930089  
1            0.004821  0.001717    0.479364   99.442240  
2            0.003162  0.001475    0.503956  159.377916  
3            0.002927  0.001475    0.466479  159.377916  
4            0.002877  0.001037    0.484193  168.298295  
                          antecedents             consequents  \
16  frozenset({214835747, 214587712})  frozenset({214835017})   
17  frozenset({214835017, 214587712})  frozenset({214835747})   
22  frozenset({214

In [12]:
clicks_rules["pair"] = clicks_rules.apply(
    lambda r: tuple(sorted([
        next(iter(r["antecedents"])),
        next(iter(r["consequents"]))
    ])),
    axis=1
)
pair_counts = clicks_rules["pair"].value_counts()
asymmetric_pairs = pair_counts[pair_counts == 1]
asymmetric_pairs

pair
(214835109, 214838227)    1
(214819760, 214819762)    1
(214827007, 214834865)    1
(214827005, 214834865)    1
(214827000, 214827005)    1
(214829878, 214831948)    1
(214587712, 214835017)    1
(214587712, 214835747)    1
(214829878, 214831946)    1
(214826610, 214834880)    1
(214834877, 214834880)    1
(214826610, 214834877)    1
(214829878, 214829882)    1
(214829765, 214835167)    1
(214836924, 214836932)    1
(214826977, 214831959)    1
(214829878, 214829887)    1
(214839971, 214839973)    1
(214826925, 214827005)    1
(214587317, 214602598)    1
(214826610, 214829387)    1
(214826610, 214833755)    1
(214829878, 214829885)    1
(214684513, 214839373)    1
Name: count, dtype: int64

The strategy for the mono-directional rules is the same as before, but with clicks. Firstly, testing the hypothesis that the two products are likely to be bought together through an A/B test. If the test confirms the hypothesis, the strategy of before can be applied in both directions.